# 00G — Run code with existing managed images

**Can SageMaker run an existing image without Docker in this notebook?**
This notebook creates two independent development pipelines, each containing
one small **billable Processing job**: one uses an AWS-provided scikit-learn
image, and one uses an image from a matching platform repository.

Paste the **exact full IAM role ARN** that worked for pipeline creation,
including its path. This notebook never constructs an ARN from STS, calls IAM,
or changes your supplied role. If the platform provides a separate job role,
enter that too. Otherwise the diagnostic tries the supplied role for both;
production role separation is a separate requirement.

Run the read/definition cells first. Enable either job independently in its
create cell. No Docker, image build, repository creation, CDK, input data or
S3 write is required. Code travels in the small Processing entrypoint command;
SageMaker pulls the existing image. Success does not establish organizational
approval to use an image. Run only image tests permitted by your platform.

Start with a fresh kernel. Each enabled test creates one execution; retries
reuse its token. To try changed code in a new execution, use a fresh kernel.
Outputs may contain account/resource names; clear them before sharing.


## 1. Exact role and image discovery settings

- Paste the full working IAM role ARN once; keep every path component.
- Find a platform repository by a name fragment, or supply an exact image URI.
- Initialize clients and unique names; use only boto3 and the standard library.


In [ ]:
import json
import os
import shutil
import subprocess
import time
import uuid
from pathlib import Path
from pprint import pprint

import boto3
from botocore.config import Config

pipeline_role_arn = ""      # Exact ARN from the working pipeline/platform team; includes role path.
job_role_arn = ""           # Empty uses that same supplied role for this diagnostic only.
repository_contains = "model-build"  # Change the fragment if needed; matching ignores case.
platform_image_uri = ""    # Optional exact image URI; otherwise discover a matching repository.
name_prefix = "imagecheck" # Adjust only if your platform requires a particular resource prefix.

session = boto3.Session()
region = session.region_name or "us-east-1"
config = Config(connect_timeout=5, read_timeout=15, retries={"max_attempts": 0})
sm = session.client("sagemaker", region_name=region, config=config)
ecr = session.client("ecr", region_name=region, config=config)
logs = session.client("logs", region_name=region, config=config)
sts = session.client("sts", region_name=region, config=config)
token = uuid.uuid4().hex
probe_name = name_prefix + "-" + token[:6]
aws_name, platform_name = probe_name + "-aws", probe_name + "-platform"
aws_pipeline = platform_pipeline = None
aws_execution = platform_execution = None
selected_repository = None
job_names, statuses, observations = {}, {}, {}
started_at = int(time.time() * 1000)
print("Region:", region, "(fallback)" if not session.region_name else "")
print("Pipeline role:", pipeline_role_arn or "paste the exact working ARN above")
print("Job role:", job_role_arn or pipeline_role_arn or "not supplied")
print("Probe names:", aws_name, platform_name)


## 2. Actual caller — GetCallerIdentity

- Print the notebook caller, which is distinct from the role passed to SageMaker.
- Keep the supplied pipeline/job ARNs unchanged.
- Treat an STS assumed-role session ARN as an identity, not a full IAM role ARN.


In [ ]:
try:
    caller = sts.get_caller_identity()
    pprint({key: caller[key] for key in ("Account", "Arn")})
except Exception as error:
    print(error)


## 3. Docker CLI, daemon binary and Studio socket

- Inspect this machine without installing or starting any services.
- If Studio exposes its proxy socket, test that endpoint explicitly with API 1.41.
- A missing proxy requires platform/domain investigation; it does not block the managed jobs below.


In [ ]:
print("Docker CLI:", shutil.which("docker"))
print("dockerd binary:", shutil.which("dockerd"))
print("DOCKER_HOST:", os.environ.get("DOCKER_HOST", "not set"))
print("Studio proxy exists:", Path("/docker/proxy.sock").exists())
print("Default socket exists:", Path("/var/run/docker.sock").exists())
try:
    response = subprocess.run(["docker", "--version"], capture_output=True, text=True, timeout=10, check=True)
    print(response.stdout)
except Exception as error:
    print(getattr(error, "stderr", "") or error)
if Path("/docker/proxy.sock").exists():
    try:
        response = subprocess.run(
            ["docker", "--host", "unix:///docker/proxy.sock", "info",
             "--format", "{{.ServerVersion}} | {{.OSType}}"],
            env={**os.environ, "DOCKER_API_VERSION": "1.41"},
            capture_output=True, text=True, timeout=15, check=True,
        )
        print(response.stdout)
    except Exception as error:
        print(getattr(error, "stderr", "") or error)
else:
    print("Studio Docker proxy not found. No daemon was started or settings changed.")


## 4. Studio Docker setting — DescribeDomain

- Read DomainId from local Studio metadata if available.
- Ask SageMaker for the configured Docker settings without calling IAM.
- A denied describe or missing field leaves the setting unknown.


In [ ]:
metadata_path = Path("/opt/ml/metadata/resource-metadata.json")
domain_id = os.environ.get("SAGEMAKER_DOMAIN_ID")
try:
    if metadata_path.exists():
        domain_id = json.loads(metadata_path.read_text()).get("DomainId") or domain_id
    if domain_id:
        response = sm.describe_domain(DomainId=domain_id)
        pprint({"DomainId": domain_id, "NetworkAccess": response.get("AppNetworkAccessType"),
                "DockerSettings": response.get("DomainSettings", {}).get("DockerSettings", "not returned; unknown")})
    else:
        print("No local Studio DomainId found; not exercised.")
except Exception as error:
    print(error)


Studio's supported setup uses a domain-enabled Docker service and the exposed
socket. Installing the CLI alone does not enable it. Domain changes apply to
newly created applications. See the
[AWS Studio Docker setup](https://docs.aws.amazon.com/sagemaker/latest/dg/studio-updated-local-get-started.html).
The cells above do not start a nested daemon or change domain/network controls.


## 5. Find matching platform repositories

- Read repository pages and filter names using the supplied fragment.
- Automatically select only a unique match; print all matches if ambiguous.
- Skip discovery if an exact platform image URI was already supplied.


In [ ]:
if not platform_image_uri:
    try:
        matches = []
        page_token = None
        while True:
            response = ecr.describe_repositories(
                maxResults=100, **({"nextToken": page_token} if page_token else {})
            )
            matches.extend(item for item in response.get("repositories", [])
                           if repository_contains.lower() in item["repositoryName"].lower())
            page_token = response.get("nextToken")
            if not page_token:
                break
        pprint([item["repositoryUri"] for item in matches])
        if len(matches) == 1:
            selected_repository = matches[0]
        else:
            print("No unique match. Narrow repository_contains or supply platform_image_uri in cell 1.")
    except Exception as error:
        print(error)
else:
    print("Using supplied image:", platform_image_uri)


## 6. Select a tagged image and pin its digest

- Read tagged images from the unique repository match.
- Print the five most recently pushed and select the newest by digest.
- A matching repository name is a candidate, not proof of runtime compatibility or approval.


In [ ]:
if not platform_image_uri and selected_repository:
    try:
        images = []
        page_token = None
        while True:
            response = ecr.describe_images(
                repositoryName=selected_repository["repositoryName"], filter={"tagStatus": "TAGGED"},
                maxResults=100, **({"nextToken": page_token} if page_token else {})
            )
            images.extend(response.get("imageDetails", []))
            page_token = response.get("nextToken")
            if not page_token:
                break
        images.sort(key=lambda item: item["imagePushedAt"], reverse=True)
        pprint([{"tags": item.get("imageTags"), "digest": item["imageDigest"],
                 "pushed": item["imagePushedAt"]} for item in images[:5]])
        if images:
            platform_image_uri = selected_repository["repositoryUri"] + "@" + images[0]["imageDigest"]
        else:
            print("The repository has no tagged images. Creation of a repository does not populate it.")
    except Exception as error:
        print(error)
print("Platform image:", platform_image_uri or "not discovered or supplied")


## 7. AWS-provided scikit-learn image

- Use the published AWS scikit-learn registry map for the current region.
- Print the existing image URI; no Docker pull or image copy is performed locally.
- Test this image only if its use is permitted by your platform.


In [ ]:
aws_image_registries = {
    "af-south-1": "510948584623",
    "ap-east-1": "651117190479",
    "ap-northeast-1": "354813040037",
    "ap-northeast-2": "366743142698",
    "ap-northeast-3": "867004704886",
    "ap-south-1": "720646828776",
    "ap-south-2": "628508329040",
    "ap-southeast-1": "121021644041",
    "ap-southeast-2": "783357654285",
    "ap-southeast-3": "951798379941",
    "ap-southeast-4": "106583098589",
    "ca-central-1": "341280168497",
    "ca-west-1": "190319476487",
    "eu-central-1": "492215442770",
    "eu-central-2": "680994064768",
    "eu-north-1": "662702820516",
    "eu-south-1": "978288397137",
    "eu-south-2": "104374241257",
    "eu-west-1": "141502667606",
    "eu-west-2": "764974769150",
    "eu-west-3": "659782779980",
    "il-central-1": "898809789911",
    "me-central-1": "272398656194",
    "me-south-1": "801668240914",
    "sa-east-1": "737474898029",
    "us-east-1": "683313688378",
    "us-east-2": "257758044811",
    "us-west-1": "746614075791",
    "us-west-2": "246618743249"
}
registry = aws_image_registries.get(region)
aws_image_uri = f"{registry}.dkr.ecr.{region}.amazonaws.com/sagemaker-scikit-learn:1.2-1-cpu-py3" if registry else None
print("AWS image:", aws_image_uri or "no registry mapping for this region")


## 8. The actual Python code to run

- Start with Python, module discovery and a simple calculation.
- For a second run, use a fresh kernel and select sklearn_probe to fit a tiny model.
- Encode the displayed code as one short command argument; full source uses S3 inputs.


In [ ]:
python_probe = """import sys,importlib.util
print('PYTHON',sys.version,flush=True)
for name in ('sklearn','pandas','numpy','torch','autogluon'):
 print(name,bool(importlib.util.find_spec(name)),flush=True)
print('IMAGE_PROBE_OK',2+2,flush=True)
"""
sklearn_probe = """from sklearn.linear_model import LinearRegression
m=LinearRegression().fit([[0],[1],[2]],[1,3,5])
print('PREDICTION',m.predict([[3]]).tolist(),flush=True)
print('IMAGE_PROBE_OK',flush=True)
"""
probe_code = python_probe  # Change to sklearn_probe for the tiny fit/predict test.
# The API requires a single-line argument, at most 256 characters.
entrypoint_command = "exec(" + repr(probe_code) + ")"
assert len(entrypoint_command) <= 256, "Use an S3 script input for a longer program."
print(probe_code)


## 9. Job resources and network settings

- Use a small CPU Processing instance and a short runtime limit.
- Keep container network isolation enabled; optionally supply platform-required VPC/key bindings.
- Print the exact role and job settings that will be sent.


In [ ]:
instance_type = "ml.m5.large"
python_executable = "python3"  # Change only if the selected image uses a different executable path.
subnet_ids = []
security_group_ids = []
volume_kms_key_id = ""

network_config = {"EnableNetworkIsolation": True}
if subnet_ids or security_group_ids:
    assert subnet_ids and security_group_ids, "Supply both subnet and security-group IDs."
    network_config["VpcConfig"] = {"Subnets": subnet_ids, "SecurityGroupIds": security_group_ids}
cluster = {"InstanceCount": 1, "InstanceType": instance_type, "VolumeSizeInGB": 10}
if volume_kms_key_id:
    cluster["VolumeKmsKeyId"] = volume_kms_key_id
job_settings = {
    "RoleArn": job_role_arn or pipeline_role_arn,
    "ProcessingResources": {"ClusterConfig": cluster},
    "NetworkConfig": network_config,
    "StoppingCondition": {"MaxRuntimeInSeconds": 300},
}
pprint(job_settings)


## 10. Create the AWS-image pipeline

- Set RUN_AWS_IMAGE to true to create this test; the next cell starts its billable job.
- Pass the exact supplied pipeline role and the visible one-step definition.
- Do not depend on the other image test succeeding.


In [ ]:
RUN_AWS_IMAGE = False
aws_definition = {
    "Version": "2020-12-01",
    "Steps": [{
        "Name": "RunCode", "Type": "Processing",
        "Arguments": {
            **job_settings, "ProcessingJobName": aws_name,
            "AppSpecification": {
                "ImageUri": aws_image_uri,
                "ContainerEntrypoint": [python_executable, "-c", entrypoint_command],
            },
        },
    }],
}
print("Pipeline role:", pipeline_role_arn, "| job role:", job_settings["RoleArn"])
print("Image:", aws_image_uri, "| pipeline:", aws_name)
if RUN_AWS_IMAGE:
    try:
        assert pipeline_role_arn.startswith("arn:") and ":iam::" in pipeline_role_arn, "Paste the exact IAM role ARN in cell 1."
        assert aws_image_uri, "No image was selected."
        response = sm.create_pipeline(
            PipelineName=aws_name, RoleArn=pipeline_role_arn,
            PipelineDefinition=json.dumps(aws_definition),
            ClientRequestToken=token + "-aws-create",
        )
        aws_pipeline = response["PipelineArn"]
        print(aws_pipeline)
        observations["AWS create"] = "confirmed"
    except Exception as error:
        print(error)
        observations["AWS create"] = str(error)
else:
    print("Not exercised: RUN_AWS_IMAGE is false.")


## 11. Start the AWS-image pipeline

- Start only the new pipeline created above.
- Reuse this operation's request token to avoid duplicate executions on cell retries.
- Job creation, image pull and execution results are checked separately below.


In [ ]:
if RUN_AWS_IMAGE and aws_pipeline:
    try:
        response = sm.start_pipeline_execution(
            PipelineName=aws_name, PipelineExecutionDisplayName="image-probe",
            ClientRequestToken=token + "-aws-start",
        )
        aws_execution = response["PipelineExecutionArn"]
        print(aws_execution)
        observations["AWS start"] = "confirmed"
    except Exception as error:
        print(error)
        observations["AWS start"] = str(error)
else:
    print("Not exercised: this probe was not created/enabled.")


## 12. Create the platform-image pipeline

- Set RUN_PLATFORM_IMAGE to true to create this test; the next cell starts its billable job.
- Pass the exact supplied pipeline role and the visible one-step definition.
- Do not depend on the other image test succeeding.


In [ ]:
RUN_PLATFORM_IMAGE = False
platform_definition = {
    "Version": "2020-12-01",
    "Steps": [{
        "Name": "RunCode", "Type": "Processing",
        "Arguments": {
            **job_settings, "ProcessingJobName": platform_name,
            "AppSpecification": {
                "ImageUri": platform_image_uri,
                "ContainerEntrypoint": [python_executable, "-c", entrypoint_command],
            },
        },
    }],
}
print("Pipeline role:", pipeline_role_arn, "| job role:", job_settings["RoleArn"])
print("Image:", platform_image_uri, "| pipeline:", platform_name)
if RUN_PLATFORM_IMAGE:
    try:
        assert pipeline_role_arn.startswith("arn:") and ":iam::" in pipeline_role_arn, "Paste the exact IAM role ARN in cell 1."
        assert platform_image_uri, "No image was selected."
        response = sm.create_pipeline(
            PipelineName=platform_name, RoleArn=pipeline_role_arn,
            PipelineDefinition=json.dumps(platform_definition),
            ClientRequestToken=token + "-platform-create",
        )
        platform_pipeline = response["PipelineArn"]
        print(platform_pipeline)
        observations["platform create"] = "confirmed"
    except Exception as error:
        print(error)
        observations["platform create"] = str(error)
else:
    print("Not exercised: RUN_PLATFORM_IMAGE is false.")


## 13. Start the platform-image pipeline

- Start only the new pipeline created above.
- Reuse this operation's request token to avoid duplicate executions on cell retries.
- Job creation, image pull and execution results are checked separately below.


In [ ]:
if RUN_PLATFORM_IMAGE and platform_pipeline:
    try:
        response = sm.start_pipeline_execution(
            PipelineName=platform_name, PipelineExecutionDisplayName="image-probe",
            ClientRequestToken=token + "-platform-start",
        )
        platform_execution = response["PipelineExecutionArn"]
        print(platform_execution)
        observations["platform start"] = "confirmed"
    except Exception as error:
        print(error)
        observations["platform start"] = str(error)
else:
    print("Not exercised: this probe was not created/enabled.")


## 14. Execution status — DescribePipelineExecution

- Inspect each started execution independently, including its failure reason.
- Poll every 15 seconds for at most 15 minutes; set the switch false for one status check.
- No new jobs are started; rerun later if the wait ends before completion.


In [ ]:
WAIT_FOR_COMPLETION = True
deadline = time.monotonic() + 900
while True:
    pending = False
    for label, arn in (("AWS", aws_execution), ("platform", platform_execution)):
        if not arn:
            print(label, "not started")
            continue
        try:
            response = sm.describe_pipeline_execution(PipelineExecutionArn=arn)
            statuses[label] = response["PipelineExecutionStatus"]
            print(label, statuses[label], response.get("FailureReason", ""), flush=True)
            pending |= statuses[label] in ("Executing", "Stopping")
        except Exception as error:
            print(label, error)
            observations[label + " status read"] = str(error)
    if not WAIT_FOR_COMPLETION or not pending or time.monotonic() >= deadline:
        break
    time.sleep(15)


## 15. Actual jobs — ListPipelineExecutionSteps

- Inspect step metadata even if describing an execution failed.
- Print step failure reasons such as a denied CreateProcessingJob or job-role PassRole.
- Record the actual Processing job name rather than guessing SageMaker's generated suffix.


In [ ]:
for label, arn in (("AWS", aws_execution), ("platform", platform_execution)):
    if not arn:
        print(label, "not started")
        continue
    try:
        response = sm.list_pipeline_execution_steps(PipelineExecutionArn=arn)
        for step in response.get("PipelineExecutionSteps", []):
            pprint({"image_test": label, "step": step["StepName"],
                    "status": step["StepStatus"], "failure": step.get("FailureReason"),
                    "metadata": step.get("Metadata")})
            job_arn = step.get("Metadata", {}).get("ProcessingJob", {}).get("Arn")
            if job_arn:
                job_names[label] = job_arn.rsplit("/", 1)[-1]
    except Exception as error:
        print(label, error)
        observations[label + " steps read"] = str(error)


## 16. Actual image and runtime role — DescribeProcessingJob

- Inspect each job that SageMaker actually created.
- Print the runtime role, image, command, status and failure reason.
- Distinguish failure to pull an image from failure to start Python or import a package.


In [ ]:
for label in ("AWS", "platform"):
    if label not in job_names:
        print(label, "no job discovered; rerun the steps cell after execution progresses.")
        continue
    try:
        response = sm.describe_processing_job(ProcessingJobName=job_names[label])
        pprint({"image_test": label, "job": job_names[label],
                "status": response["ProcessingJobStatus"], "failure": response.get("FailureReason"),
                "role": response["RoleArn"], "application": response["AppSpecification"]})
        observations[label + " job"] = response["ProcessingJobStatus"]
    except Exception as error:
        print(label, error)
        observations[label + " job read"] = str(error)


## 17. Actual container output — FilterLogEvents

- Read CloudWatch logs only for these job names and this run's time window.
- Look for Python/package output and IMAGE_PROBE_OK, not just a successful start response.
- Rerun if logs have not arrived; persistent missing streams also warrant checking the job role's logging permissions.


In [ ]:
for label in ("AWS", "platform"):
    if label not in job_names:
        print(label, "no job discovered")
        continue
    print("\n---", label, job_names[label], "---")
    try:
        page_token = None
        messages = []
        for page in range(5):
            response = logs.filter_log_events(
                logGroupName="/aws/sagemaker/ProcessingJobs",
                logStreamNamePrefix=job_names[label] + "/",
                startTime=started_at - 60000, endTime=int(time.time() * 1000) + 60000,
                limit=100, **({"nextToken": page_token} if page_token else {}),
            )
            for event in response.get("events", []):
                print(event["message"])
                messages.append(event["message"])
            next_token = response.get("nextToken")
            if not next_token or next_token == page_token:
                break
            page_token = next_token
        found = any(line.strip() in ("IMAGE_PROBE_OK", "IMAGE_PROBE_OK 4")
                    for message in messages for line in message.splitlines())
        observations[label + " output"] = "IMAGE_PROBE_OK observed" if found else "marker not observed; inspect/rerun logs"
        print(observations[label + " output"])
    except Exception as error:
        print(label, error)
        observations[label + " output"] = str(error)


## 18. One summary to capture

- Show exact roles/images and observed outcomes in one place.
- Keep the AWS and platform results independent.
- Save full original errors from the cells when requesting access or image changes.


In [ ]:
print("MANAGED IMAGE CHECK")
print("Region:", region)
print("Pipeline role:", pipeline_role_arn)
print("Job role:", job_settings["RoleArn"])
print("AWS image:", aws_image_uri)
print("Platform image:", platform_image_uri or "none")
for label, arn in (("AWS", aws_execution), ("platform", platform_execution)):
    print(label, "| execution:", statuses.get(label, "not observed"), "|", arn or "not started")
for name, outcome in observations.items():
    print(name + ":", " ".join(str(outcome).split())[:180])
print("Not tested: S3 access, package downloads, custom-image build/push, Training or Batch Transform.")
print("Module discovery means presence only; use sklearn_probe to test a tiny fit/predict.")


## 19. Optional: stop this notebook's running executions

- Enable only if you want to stop these probes before they finish.
- Inspect status first and stop only Executing executions created in this kernel.
- Wait for a terminal state before deleting pipelines.


In [ ]:
STOP_PROBES = False
if STOP_PROBES:
    for arn in (aws_execution, platform_execution):
        if arn:
            try:
                response = sm.describe_pipeline_execution(PipelineExecutionArn=arn)
                if response["PipelineExecutionStatus"] == "Executing":
                    print(sm.stop_pipeline_execution(PipelineExecutionArn=arn))
                else:
                    print(arn, response["PipelineExecutionStatus"])
            except Exception as error:
                print(error)
else:
    print("Stop is off.")


## 20. Optional: delete this notebook's pipelines

- Enable after the executions finish.
- Delete only the successfully created probe pipelines from this kernel.
- Keep job history and logs; never delete image repositories or application pipelines.


In [ ]:
DELETE_PROBES = False
if DELETE_PROBES:
    for label, arn, name in (("aws", aws_pipeline, aws_name), ("platform", platform_pipeline, platform_name)):
        if arn:
            try:
                sm.delete_pipeline(PipelineName=name, ClientRequestToken=token + "-" + label + "-delete")
                print("Deleted:", arn)
            except Exception as error:
                print(error)
else:
    print("Cleanup is off; pipeline names:", aws_name, platform_name)


## What a successful test enables

A successful job plus `IMAGE_PROBE_OK` shows that the runtime role can use that
image and run the displayed Python command. For the default probe, the package
lines show only whether modules were found. Selecting `sklearn_probe` instead
tests a tiny fit/predict operation whose expected prediction is approximately 7.

For full source files, the next step is an **approved S3 prefix**. A Processing
input can download a source folder to `/opt/ml/processing/input/code`, then
the entrypoint can run `python3 /opt/ml/processing/input/code/entrypoint.py`.
Python imports can use the other files in that folder. No image rebuild is
needed when the required dependencies are already in the image.

Changing installed dependencies or the image filesystem requires an approved
packaging/build route. Copying an unchanged image to another ECR repository
would not install your code or dependencies. A managed Processing success also
does not establish the separate Training or Batch Transform container contracts.
The application's current code-in-image packaging remains unchanged by this probe.

References:
[Processing entrypoint behavior](https://docs.aws.amazon.com/sagemaker/latest/dg/byoc-run-image.html),
[Processing inputs](https://docs.aws.amazon.com/sagemaker/latest/APIReference/API_ProcessingS3Input.html),
[IAM role paths and STS session ARNs](https://docs.aws.amazon.com/IAM/latest/UserGuide/reference_identifiers.html),
[AWS scikit-learn image map](https://github.com/aws/sagemaker-python-sdk/blob/v2.253.1/src/sagemaker/image_uri_config/sklearn.json).
